# Nettoyage backtest — du corpus brut aux tables de recherche

**La logique vit dans le code** : `common/backtest_clean.py`, exécuté par le pipeline en step 7
(`python -m common.pipeline …` → « Nettoyage — tables de recherche »), testé par
`tests/regression/test_backtest_clean.py` (ZÉRO ÉCART exigé). **Ce notebook est sa vitrine** :
il montre l'entonnoir, les corrections et les trois tables — il ne porte aucune logique propre.

Trois sorties dans `data/clean/` :

| table | contenu |
|---|---|
| `transactions_brut_2014_2026.csv` | **tout** le corpus assemblé (dédup cross-année + fixes read-time) — chaque ligne porte le **verdict** de l'entonnoir (`exclusion_etape`, `exclusion_motif`) : rien d'écarté en silence |
| `transactions_backtest_2014_2026.csv` | la table **clean** (entonnoir A→D + corrections) — celle que la recherche consomme |
| `transactions_gated_2014_2026.csv` | les transactions des scans manuscrits **gated** (politique `house/ocr.py`), récupérées par une passe OCR une-fois — exportées avec leur motif |

⚠️ **La recherche publiée (notebooks 05b→16, fiches) est adossée à la version du 2026-07-04**,
archivée dans `_archive/data_clean/transactions_backtest_2014_2026_v20260704.csv` (134 464 × 36).
La table courante intègre les corrections du 2026-08-11 — le diff complet est chiffré dans
`NOTE_DIFF_TABLE_CLEAN.md`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

def _find_repo():
    here = Path.cwd()
    cands = [here, *here.parents, here / "00_recuperation_donnees",
             Path.home() / "Downloads" / "Jupiter" / "00_recuperation_donnees"]
    for c in cands:
        if (c / "common" / "backtest_clean.py").exists() and (c / "data" / "house" / "tables").exists():
            return c
    raise RuntimeError("Racine (common/ + data/) introuvable")

REPO = _find_repo(); sys.path.insert(0, str(REPO))
from common import backtest_clean as bc
print("Module :", bc.__file__)
brut, clean, gated, funnel = bc.build_tables(corrections=True)

Module : /Users/lemairealice/Downloads/Jupiter/00_recuperation_donnees/common/backtest_clean.py


169,000 transactions uniques chargées (load_final)


tickers faux-positifs corrigés (read-time) : 152 lignes


entonnoir : {'A': np.int64(3252), 'B': np.int64(29783), 'C': np.int64(826), 'D': np.int64(687), '∅ (gardée)': np.int64(134452)}


## 1. L'entonnoir A→D — et le brut qui garde tout

Quatre coupes, dans l'ordre, une seule cause par ligne. La table **brute** conserve les
34 548 lignes écartées **avec leur motif** : la décision d'en juger autrement revient à la
recherche, pas au nettoyage.

In [2]:
recap = (brut["exclusion_etape"].replace("", "∅ gardée (→ table clean)")
         .value_counts().rename("lignes").to_frame()
         .join(brut.groupby(brut["exclusion_etape"].replace("", "∅ gardée (→ table clean)"))["exclusion_motif"]
                   .agg(lambda s: " · ".join(sorted(set(x for x in s if x))))))
display(recap)
assert int(recap.loc["∅ gardée (→ table clean)", "lignes"]) == len(clean)
print(f"{len(clean):,} lignes clean = {100*len(clean)/len(brut):.1f} % du brut ({len(brut):,})")

,lignes,exclusion_motif
exclusion_etape,,
∅ gardée (→ table clean),134452,
B,29783,option / obligation (famille non cotée) · tick...
A,3252,dates absentes ou incohérentes (divulgation < ...
C,826,"direction hors {achat, vente} (échange, autre)"
D,687,montant absent


134,452 lignes clean = 79.6 % du brut (169,000)


## 2. Les corrections (2026-08-11) — des référentiels, pas du code ad hoc

- **`ticker_false_positives.csv`** — les tickers extraits à tort d'une parenthèse descriptive
  (« NetApp (IRA) » → le ticker n'est pas IRA ; « FACEBOOK CL-A » → le ticker n'est pas A) :
  corrigés vers le vrai symbole quand il existe, vidés sinon ;
- **`ticker_renames.csv` complété (+12)** — les renommages continus que la recherche rattrapait à
  la main (`MMC→MRSH`, `NLOK→GEN`, `CNHI→CNH`…), chacun **vérifié sur cours** (2026-08-11) ;
  et la ligne `FISV` remise dans le bon sens (Yahoo sert la série sous FISV, pas sous FI) ;
- **`ticker_sector_map.csv`** — les 8 fonds que la table contredisait (EDR = un REIT, pas de la
  « Communication » ; FDN/VNQ/… = des ETF, pas des actions) ;
- **`ticker_sector_map_datee.csv`** — la carte sectorielle **datée** (bascules XLRE 2016 / XLC
  2018 + 45 attributions nominatives), sortie du notebook 16 vers le référentiel ;
- **sous-commissions résolues et commissions normalisées** — `HSAG15` devient « House Committee
  on Agriculture — Conservation, Energy, and Forestry » ; le texte vit dans la table annexe
  `commissions_membre_congres.csv` (jointure `bioguide_id × congress`) — l'inliner sur chaque
  ligne aurait décuplé le poids des tables ; la ligne garde `congress` + `committees_key_flag`.

In [3]:
fp = pd.read_csv(bc.REF / "ticker_false_positives.csv")
ren = pd.read_csv(bc.REF / "ticker_renames.csv")
print(f"règles faux-positifs : {len(fp)} | renommages au référentiel : {len(ren)}")
display(fp.head(8))
ex = clean.loc[clean["ticker_source"] == "false_positive_fixed",
               ["member_name", "ticker", "ticker_yahoo", "asset_description"]]
print(f"lignes corrigées présentes en clean : {len(ex)}")
display(ex.drop_duplicates("asset_description").head(8))
com = pd.read_csv(REPO / "data" / "clean" / "commissions_membre_congres.csv")
import re
n_raw = int(com["committee_membership"].astype(str)
            .str.contains(r"\b(?:H|S|J)S[A-Z]{2}\d{2}\b", regex=True).sum())
print(f"table annexe commissions : {com.shape} | codes bruts restants : {n_raw} (attendu : 0)")

règles faux-positifs : 26 | renommages au référentiel : 96


,ticker_errone,motif_description,ticker_corrige,note
0,IRA,net app,NTAP,"« (IRA) » = le compte, pas le titre"
1,IRA,netapp,NTAP,NaN
2,IRA,innate immunotherapeutics,NaN,société australienne non cotée aux US — pas de...
3,IRA,primecap odyssey,POAGX,fonds commun
4,IRA,spdr msci acwi ex-us,CWI,NaN
5,ADR,shell plc,SHEL,"« ADR » = la forme du titre, pas le symbole"
6,ADR,novartis,NVS,NaN
7,REIT,host hotels,HST,"« (REIT) » = la forme, pas le symbole"


lignes corrigées présentes en clean : 133


,member_name,ticker,ticker_yahoo,asset_description
3736,Rodney P. Frelinghuysen,CWI,CWI,SPDR MSCI ACWI EX-US (IRA)
3737,Rodney P. Frelinghuysen,POAGX,POAGX,PRIMECAP ODYSSEY AGGR (IRA)
8804,James B. Renacci,CMD,STE,Cantel Medical Corporation (CMN)
9070,James B. Renacci,CMD,STE,CANTEL MEDICAL CORPORATION (CMN)
10232,Bradley S. Schneider,DATA,CRM,TABLEAU SOFTWARE INC CL-A
10687,Ed Whitfield,RDS-B,SHEL,Royal Dutch Shell-B
17672,Tom Price,FB,META,Facebook Inc CL-A
22296,Thomas R Tillis,CMD,STE,Cantel Medical Corp. (NYSE)


table annexe commissions : (3707, 3) | codes bruts restants : 0 (attendu : 0)


## 3. Les trois tables — aperçus

In [4]:
for nom, t in [("brut", brut), ("clean", clean), ("gated", gated)]:
    print(f"{nom:5s} : {t.shape[0]:,} × {t.shape[1]}")
cols_apercu = ["member_name", "party", "ticker", "ticker_yahoo", "direction", "amount_midpoint",
               "transaction_date", "asset_class", "owner_n", "ticker_groupe"]
display(clean[cols_apercu].head(5))
display(gated.head(5))

brut  : 169,000 × 41
clean : 134,452 × 39
gated : 7,287 × 13


,member_name,party,ticker,ticker_yahoo,direction,amount_midpoint,transaction_date,asset_class,owner_n,ticker_groupe
0,Bob Gibbs,Republican,INTC,INTC,buy,8000.5,2014-12-30,stock,élu,INTC
1,Bob Gibbs,Republican,B,B,sell,8000.5,2014-12-23,stock,élu,B
2,Bob Gibbs,Republican,AAPL,AAPL,buy,8000.5,2014-12-30,stock,élu,AAPL
7,Lois Frankel,Democrat,STJ,STJ,buy,8000.5,2014-12-03,stock,élu,STJ
8,Lois Frankel,Democrat,MMM,MMM,sell,8000.5,2014-12-10,stock,élu,MMM


,member_name,party,owner,ticker,direction,amount_midpoint,amount_code,transaction_date,disclosure_date,doc_id,gating_cluster,gating_reason,lag_days
0,James Comer,Republican,Self,CTBI,buy,8000.0,A,2020-02-21,2020-03-25,8217113,C_manuscrit,scan manuscrit — politique de gating house/ocr...,33
1,George Holding,Republican,Self,JNJ,sell,8000.0,A,2020-02-26,2020-03-12,8217083,C_manuscrit,scan manuscrit — politique de gating house/ocr...,15
2,George Holding,Republican,Self,MSFT,sell,8000.0,A,2020-02-26,2020-03-12,8217083,C_manuscrit,scan manuscrit — politique de gating house/ocr...,15
3,George Holding,Republican,Self,IBM,sell,32500.0,B,2020-08-14,2020-10-02,8217671,C_manuscrit,scan manuscrit — politique de gating house/ocr...,49
4,Mike Kelly,Republican,Spouse,CLF,sell,32500.0,B,2020-02-05,2020-05-20,8217253,C_manuscrit,scan manuscrit — politique de gating house/ocr...,105


## 4. Conformité : ce que le pipeline a écrit ≡ ce que ce notebook vient de reconstruire

Le même hash de rendu CSV — la vitrine et le step 7 ne peuvent pas diverger sans que ce contrôle
(et le filet `test_backtest_clean.py`) le voie.

In [5]:
import hashlib
sha = lambda d: hashlib.sha256(d.to_csv(index=False).encode()).hexdigest()
for nom, t, f in [("brut", brut, "transactions_brut_2014_2026.csv"),
                  ("clean", clean, "transactions_backtest_2014_2026.csv"),
                  ("gated", gated, "transactions_gated_2014_2026.csv")]:
    disque = (REPO / "data" / "clean" / f).read_bytes()
    ok = hashlib.sha256(disque).hexdigest() == sha(t)
    print(f"{nom:5s} : {'✅ identique au fichier écrit par le pipeline' if ok else '❌ DIVERGENCE'}")
    assert ok

brut  : ✅ identique au fichier écrit par le pipeline


clean : ✅ identique au fichier écrit par le pipeline
gated : ✅ identique au fichier écrit par le pipeline


## 5. Le diff v1 → courante, en résumé

Détail correction par correction : `NOTE_DIFF_TABLE_CLEAN.md`. Les invariants de la table clean
(bioguide, ticker, montant, direction, chronologie, hash) sont assertés **dans le module** à
chaque construction.

In [6]:
v1 = pd.read_csv(REPO / "_archive" / "data_clean" / "transactions_backtest_2014_2026_v20260704.csv",
                 low_memory=False)
print(f"v1 : {v1.shape} → courante : {clean.shape}")
k = ["natural_key_hash", "occurrence_index", "doc_id"]
v1k = set(map(tuple, v1[k].astype(str).values)); v2k = set(map(tuple, clean[k].astype(str).values))
print(f"sorties (tickers faux-positifs devenus vides) : {len(v1k - v2k)} | entrées : {len(v2k - v1k)}")
m1 = v1.set_index([v1[c].astype(str) for c in k]); m2 = clean.set_index([clean[c].astype(str) for c in k])
com = m2.index.intersection(m1.index)
for col in ["ticker", "ticker_yahoo", "asset_class", "sector_gics", "committees_key_flag"]:
    d = int((m1.loc[com, col].astype(str) != m2.loc[com, col].astype(str)).sum())
    print(f"  {col:22s} : {d:>7,} valeurs changées")

v1 : (134464, 36) → courante : (134452, 39)
sorties (tickers faux-positifs devenus vides) : 12 | entrées : 0


  ticker                 :     133 valeurs changées
  ticker_yahoo           :     932 valeurs changées
  asset_class            :     149 valeurs changées
  sector_gics            :   3,948 valeurs changées


  committees_key_flag    :  14,850 valeurs changées
